# BrainScanAI — Préprocessing et extraction des features

## Objectifs

Cette étape vise à :

- préparer les images selon les exigences d'un modèle pré-entraîné ;
- appliquer un redimensionnement et une normalisation adaptés ;
- charger un modèle ResNet-50 pré-entraîné ;
- geler ses paramètres ;
- extraire des embeddings visuels ;
- conserver l'alignement entre chaque image, son label éventuel et son vecteur de features ;
- sauvegarder les résultats dans des tableaux exploitables.

Le modèle est utilisé comme extracteur de caractéristiques. Il n'est pas
entraîné pour établir un diagnostic médical.

## Représentations étudiées

Deux couches de ResNet-50 seront utilisées :

- `layer3` : représentation intermédiaire de 1 024 dimensions après agrégation spatiale ;
- `avgpool` : représentation finale du backbone de 2 048 dimensions.

Ces deux espaces seront comparés lors de l'analyse non supervisée et du clustering.

## 1. Imports et configuration

In [58]:
from pathlib import Path
import json
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision

from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.models.feature_extraction import create_feature_extractor
from tqdm.auto import tqdm

In [59]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f"PyTorch : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"OpenCV : {cv2.__version__}")
print("Imports réussis.")

PyTorch : 2.13.0
Torchvision : 0.28.0
OpenCV : 5.0.0
Imports réussis.


## 2. Localisation des données

Les images originales restent dans `data/raw`.

Les features générées seront placées dans `data/processed`, qui est exclu
du dépôt Git afin de ne pas versionner les données médicales ni leurs
représentations numériques.

In [60]:
def trouver_racine_projet(repertoire_depart: Path) -> Path:
    """
    Recherche la racine du projet contenant le fichier pyproject.toml.
    """

    for candidat in (
        repertoire_depart,
        *repertoire_depart.parents,
    ):
        if (candidat / "pyproject.toml").exists():
            return candidat

    raise FileNotFoundError(
        "Impossible de localiser la racine du projet."
    )


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = trouver_racine_projet(CURRENT_DIR)

DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "mri_dataset_brain_cancer_oc"
)

CANCER_DIR = DATASET_DIR / "avec_labels" / "cancer"
NORMAL_DIR = DATASET_DIR / "avec_labels" / "normal"
UNLABELED_DIR = DATASET_DIR / "sans_label"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "resnet50_imagenet1k_v2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Dataset : {DATASET_DIR}")
print(f"Sorties : {OUTPUT_DIR}")
print()

print(f"Dossier cancer : {CANCER_DIR.exists()}")
print(f"Dossier normal : {NORMAL_DIR.exists()}")
print(f"Dossier sans label : {UNLABELED_DIR.exists()}")

Racine du projet : /Users/vincentdesmouceaux/dev/BrainScanAI
Dataset : /Users/vincentdesmouceaux/dev/BrainScanAI/data/raw/mri_dataset_brain_cancer_oc
Sorties : /Users/vincentdesmouceaux/dev/BrainScanAI/data/processed/features/resnet50_imagenet1k_v2

Dossier cancer : True
Dossier normal : True
Dossier sans label : True


## 3. Inventaire des images

Chaque image est associée à :

- son chemin ;
- sa catégorie ;
- son éventuel label expert ;
- un identifiant numérique de classe.

La classe `normal` est encodée `0`, la classe `cancer` est encodée `1`
et les images non annotées conservent une valeur manquante.

In [61]:
EXTENSIONS_IMAGES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
}


def ajouter_images(
    records: list[dict],
    repertoire: Path,
    categorie: str,
    label: str | None,
    label_id: int | None,
) -> None:
    """
    Ajoute à l'inventaire toutes les images d'un répertoire.
    """

    image_paths = sorted(
        chemin
        for chemin in repertoire.iterdir()
        if chemin.is_file()
        and chemin.suffix.lower() in EXTENSIONS_IMAGES
    )

    for image_path in image_paths:
        records.append(
            {
                "path": str(image_path),
                "chemin_relatif": str(
                    image_path.relative_to(PROJECT_ROOT)
                ),
                "nom_fichier": image_path.name,
                "categorie": categorie,
                "label": label,
                "label_id": label_id,
            }
        )

In [62]:
records = []

ajouter_images(
    records=records,
    repertoire=CANCER_DIR,
    categorie="cancer",
    label="cancer",
    label_id=1,
)

ajouter_images(
    records=records,
    repertoire=NORMAL_DIR,
    categorie="normal",
    label="normal",
    label_id=0,
)

ajouter_images(
    records=records,
    repertoire=UNLABELED_DIR,
    categorie="sans_label",
    label=None,
    label_id=None,
)

df_images = pd.DataFrame(records)

df_images["label_id"] = pd.array(
    df_images["label_id"],
    dtype="Int64",
)

print(f"Nombre total d'images : {len(df_images)}")

display(
    df_images["categorie"]
    .value_counts()
    .rename_axis("categorie")
    .reset_index(name="nombre_images")
)

display(df_images.head())

Nombre total d'images : 1506


,categorie,nombre_images
0,sans_label,1406
1,cancer,50
2,normal,50


,path,chemin_relatif,nom_fichier,categorie,label,label_id
0,/Users/vincentdesmouceaux/dev/BrainScanAI/data...,data/raw/mri_dataset_brain_cancer_oc/avec_labe...,05340cd4-3bb2-459d-9937-bf27d52d8351.jpg,cancer,cancer,1
1,/Users/vincentdesmouceaux/dev/BrainScanAI/data...,data/raw/mri_dataset_brain_cancer_oc/avec_labe...,0c6f3641-60d9-4a76-abe5-de89d55d5f2c.jpg,cancer,cancer,1
2,/Users/vincentdesmouceaux/dev/BrainScanAI/data...,data/raw/mri_dataset_brain_cancer_oc/avec_labe...,0f718241-8f63-4b55-81ce-315324b51069.jpg,cancer,cancer,1
3,/Users/vincentdesmouceaux/dev/BrainScanAI/data...,data/raw/mri_dataset_brain_cancer_oc/avec_labe...,11a7a426-4806-401e-98b2-b96e7094d1a6.jpg,cancer,cancer,1
4,/Users/vincentdesmouceaux/dev/BrainScanAI/data...,data/raw/mri_dataset_brain_cancer_oc/avec_labe...,1c043dbb-4623-4769-8e5e-0223bd745040.jpg,cancer,cancer,1


## 4. Contrôle technique complet

Chaque image est ouverte et réellement décodée avant l'extraction des
features.

Les contrôles portent sur :

- la lisibilité du fichier ;
- le format réel ;
- les dimensions ;
- le mode de couleur ;
- le nombre de canaux.

L'extraction sera interrompue si une image invalide est détectée.

In [63]:
def controler_image(image_path: str) -> dict:
    """
    Contrôle la validité et les propriétés techniques d'une image.
    """

    resultat = {
        "path": image_path,
        "valide": False,
        "format_image": None,
        "largeur": np.nan,
        "hauteur": np.nan,
        "mode_couleur": None,
        "nombre_canaux": np.nan,
        "erreur": None,
    }

    try:
        with Image.open(image_path) as image:
            image.load()

            resultat.update(
                {
                    "valide": True,
                    "format_image": image.format,
                    "largeur": image.width,
                    "hauteur": image.height,
                    "mode_couleur": image.mode,
                    "nombre_canaux": len(image.getbands()),
                }
            )

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
    ) as erreur:
        resultat["erreur"] = str(erreur)

    return resultat

In [64]:
quality_records = [
    controler_image(image_path)
    for image_path in tqdm(
        df_images["path"],
        desc="Contrôle des images",
    )
]

df_quality = pd.DataFrame(quality_records)

df_images = df_images.merge(
    df_quality,
    on="path",
    how="left",
)

nombre_valides = int(df_images["valide"].sum())
nombre_invalides = int((~df_images["valide"]).sum())

print(f"Images valides : {nombre_valides}")
print(f"Images invalides : {nombre_invalides}")

Contrôle des images:   0%|          | 0/1506 [00:00<?, ?it/s]

Images valides : 1506
Images invalides : 0


In [65]:
resume_technique = (
    df_images[
        [
            "format_image",
            "largeur",
            "hauteur",
            "mode_couleur",
            "nombre_canaux",
        ]
    ]
    .value_counts()
    .reset_index(name="nombre_images")
)

display(resume_technique)

,format_image,largeur,hauteur,mode_couleur,nombre_canaux,nombre_images
0,JPEG,512,512,RGB,3,1506


In [66]:
if nombre_invalides > 0:
    display(
        df_images.loc[
            ~df_images["valide"],
            [
                "chemin_relatif",
                "erreur",
            ],
        ]
    )

    raise ValueError(
        "Des images invalides doivent être traitées "
        "avant l'extraction des features."
    )

print("Toutes les images sont exploitables.")

Toutes les images sont exploitables.


## 5. Preprocessing adapté à ResNet-50

Le preprocessing associé aux poids `IMAGENET1K_V2` est utilisé
directement afin de garantir sa compatibilité avec le modèle.

Il réalise notamment :

- le redimensionnement des images ;
- un crop central en `224 × 224` ;
- la conversion des pixels vers l'intervalle `[0, 1]` ;
- la normalisation avec les statistiques ImageNet.

Les fichiers fournis sont des JPEG, contrairement au format PNG annoncé
dans le mail initial. Cette divergence est documentée, mais n'empêche pas
leur traitement.

In [67]:
WEIGHTS = ResNet50_Weights.IMAGENET1K_V2
PREPROCESS = WEIGHTS.transforms()

print(PREPROCESS)
print()
print(f"Redimensionnement : {PREPROCESS.resize_size}")
print(f"Crop final : {PREPROCESS.crop_size}")
print(f"Moyennes : {PREPROCESS.mean}")
print(f"Écarts-types : {PREPROCESS.std}")

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

Redimensionnement : [232]
Crop final : [224]
Moyennes : [0.485, 0.456, 0.406]
Écarts-types : [0.229, 0.224, 0.225]


In [68]:
observation_test = df_images.sample(
    n=1,
    random_state=RANDOM_STATE,
).iloc[0]

with Image.open(observation_test["path"]) as image:
    image_rgb = image.convert("RGB")
    image_tensor = PREPROCESS(image_rgb)

print(f"Image originale : {image_rgb.size}")
print(f"Tensor transformé : {tuple(image_tensor.shape)}")
print(f"Type : {image_tensor.dtype}")

Image originale : (512, 512)
Tensor transformé : (3, 224, 224)
Type : torch.float32


## 6. Dataset et DataLoader

Le Dataset ouvre chaque image au moment de son utilisation, la convertit
en RGB et applique le preprocessing ResNet.

Le DataLoader traite les images par lots afin de contrôler l'utilisation
de la mémoire.

In [69]:
class BrainScanDataset(Dataset):
    """
    Dataset PyTorch utilisé pour charger les images cérébrales.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform,
    ) -> None:
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, int]:
        image_path = self.dataframe.iloc[index]["path"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_tensor = self.transform(image)

        return image_tensor, index

In [70]:
def selectionner_device() -> torch.device:
    """
    Sélectionne MPS, CUDA ou CPU selon le matériel disponible.
    """

    if (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


DEVICE = selectionner_device()

BATCH_SIZE = 32
NUM_WORKERS = 0

dataset = BrainScanDataset(
    dataframe=df_images,
    transform=PREPROCESS,
)

data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
    drop_last=False,
)

print(f"Device : {DEVICE}")
print(f"Nombre d'images : {len(dataset)}")
print(f"Nombre de batchs : {len(data_loader)}")
print(f"Taille maximale d'un batch : {BATCH_SIZE}")

Device : mps
Nombre d'images : 1506
Nombre de batchs : 48
Taille maximale d'un batch : 32


## 7. Chargement du modèle pré-entraîné

ResNet-50 est utilisé uniquement comme extracteur de caractéristiques.

Toutes ses couches sont gelées :

- aucun gradient n'est calculé ;
- aucun poids n'est modifié ;
- la couche finale de classification ImageNet n'est pas utilisée.

Les activations de `layer3` et `avgpool` sont récupérées afin de comparer
deux niveaux de représentation.

In [71]:
backbone = resnet50(
    weights=WEIGHTS,
)

for parameter in backbone.parameters():
    parameter.requires_grad = False

feature_extractor = create_feature_extractor(
    backbone,
    return_nodes={
        "layer3": "layer3",
        "avgpool": "avgpool",
    },
)

feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

nombre_parametres_entrainables = sum(
    parameter.numel()
    for parameter in feature_extractor.parameters()
    if parameter.requires_grad
)

print(
    f"Paramètres entraînables : "
    f"{nombre_parametres_entrainables:,}"
)

Paramètres entraînables : 0


In [72]:
batch_test, indices_test = next(iter(data_loader))

batch_test = batch_test[:2].to(DEVICE)

with torch.inference_mode():
    sorties_test = feature_extractor(batch_test)

layer3_test = F.adaptive_avg_pool2d(
    sorties_test["layer3"],
    output_size=(1, 1),
).flatten(start_dim=1)

avgpool_test = sorties_test[
    "avgpool"
].flatten(start_dim=1)

print(f"Entrée : {tuple(batch_test.shape)}")

print(
    f"Sortie brute layer3 : "
    f"{tuple(sorties_test['layer3'].shape)}"
)

print(
    f"Embedding layer3 : "
    f"{tuple(layer3_test.shape)}"
)

print(
    f"Sortie brute avgpool : "
    f"{tuple(sorties_test['avgpool'].shape)}"
)

print(
    f"Embedding avgpool : "
    f"{tuple(avgpool_test.shape)}"
)

Entrée : (2, 3, 224, 224)
Sortie brute layer3 : (2, 1024, 14, 14)
Embedding layer3 : (2, 1024)
Sortie brute avgpool : (2, 2048, 1, 1)
Embedding avgpool : (2, 2048)


## 8. Extraction complète des embeddings

Les images sont traitées par lots et sans calcul de gradient.

Pour chaque image, deux représentations sont extraites :

- `layer3` : vecteur intermédiaire de 1 024 dimensions ;
- `avgpool` : vecteur final de 2 048 dimensions.

Les indices retournés par le Dataset permettent de conserver
l'alignement exact entre chaque image et ses embeddings.

In [73]:
def synchroniser_device(device: torch.device) -> None:
    """
    Attend la fin des opérations du GPU avant de mesurer le temps.
    """

    if device.type == "cuda":
        torch.cuda.synchronize()

    elif device.type == "mps":
        torch.mps.synchronize()

In [74]:
def extraire_embeddings(
    loader: DataLoader,
    model: torch.nn.Module,
    device: torch.device,
) -> tuple[dict[str, np.ndarray], np.ndarray, float]:
    """
    Extrait les embeddings layer3 et avgpool pour toutes les images.
    """

    layer3_batches = []
    avgpool_batches = []
    index_batches = []

    synchroniser_device(device)
    debut = time.perf_counter()

    with torch.inference_mode():
        for images, indices in tqdm(
            loader,
            desc="Extraction des features",
        ):
            images = images.to(
                device,
                non_blocking=device.type == "cuda",
            )

            sorties = model(images)

            layer3_features = F.adaptive_avg_pool2d(
                sorties["layer3"],
                output_size=(1, 1),
            ).flatten(start_dim=1)

            avgpool_features = sorties[
                "avgpool"
            ].flatten(start_dim=1)

            layer3_batches.append(
                layer3_features
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            avgpool_batches.append(
                avgpool_features
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            index_batches.append(
                indices.numpy()
            )

    synchroniser_device(device)
    duree = time.perf_counter() - debut

    embeddings = {
        "layer3": np.concatenate(
            layer3_batches,
            axis=0,
        ),
        "avgpool": np.concatenate(
            avgpool_batches,
            axis=0,
        ),
    }

    indices = np.concatenate(
        index_batches,
        axis=0,
    )

    return embeddings, indices, duree

In [75]:
embeddings, indices, duree_extraction = extraire_embeddings(
    loader=data_loader,
    model=feature_extractor,
    device=DEVICE,
)

print(
    f"Durée totale : "
    f"{duree_extraction:.2f} secondes"
)

print(
    f"Débit moyen : "
    f"{len(df_images) / duree_extraction:.2f} images/seconde"
)

print(
    f"Shape layer3 : "
    f"{embeddings['layer3'].shape}"
)

print(
    f"Shape avgpool : "
    f"{embeddings['avgpool'].shape}"
)

Extraction des features:   0%|          | 0/48 [00:00<?, ?it/s]

Durée totale : 5.25 secondes
Débit moyen : 286.75 images/seconde
Shape layer3 : (1506, 1024)
Shape avgpool : (1506, 2048)


## 9. Alignement entre les images et les embeddings

Le DataLoader ne mélange pas les données, mais les indices sont tout de
même contrôlés et triés afin de garantir que chaque ligne de la matrice
de features correspond à la bonne image.

In [ ]:
ordre = np.argsort(indices)
indices_tries = indices[ordre]

df_metadata = (
    df_images
    .iloc[indices_tries]
    .reset_index(drop=True)
    .copy()
)

layer3_embeddings = embeddings[
    "layer3"
][ordre]

avgpool_embeddings = embeddings[
    "avgpool"
][ordre]

assert np.array_equal(
    indices_tries,
    np.arange(len(df_images)),
)

assert len(df_metadata) == len(layer3_embeddings)
assert len(df_metadata) == len(avgpool_embeddings)

print("Alignement images / métadonnées / embeddings : OK")

## 10. Contrôle de la qualité des embeddings

Les matrices de features sont contrôlées afin de vérifier :

- leurs dimensions ;
- l'absence de valeurs `NaN` ;
- l'absence de valeurs infinies ;
- l'absence de vecteurs entièrement nuls ;
- leur correspondance avec les 1 506 images.

In [ ]:
def resumer_embeddings(
    nom: str,
    tableau: np.ndarray,
) -> dict:
    """
    Produit un résumé technique d'une matrice d'embeddings.
    """

    normes = np.linalg.norm(
        tableau,
        axis=1,
    )

    return {
        "representation": nom,
        "nombre_images": tableau.shape[0],
        "nombre_features": tableau.shape[1],
        "valeurs_finies": bool(
            np.isfinite(tableau).all()
        ),
        "vecteurs_nuls": int(
            np.sum(normes == 0)
        ),
        "norme_moyenne": float(
            normes.mean()
        ),
        "norme_minimale": float(
            normes.min()
        ),
        "norme_maximale": float(
            normes.max()
        ),
    }


df_resume_embeddings = pd.DataFrame(
    [
        resumer_embeddings(
            "layer3",
            layer3_embeddings,
        ),
        resumer_embeddings(
            "avgpool",
            avgpool_embeddings,
        ),
    ]
)

display(df_resume_embeddings)

In [ ]:
assert layer3_embeddings.shape == (
    len(df_images),
    1024,
)

assert avgpool_embeddings.shape == (
    len(df_images),
    2048,
)

assert np.isfinite(
    layer3_embeddings
).all()

assert np.isfinite(
    avgpool_embeddings
).all()

assert not np.any(
    np.linalg.norm(
        layer3_embeddings,
        axis=1,
    ) == 0
)

assert not np.any(
    np.linalg.norm(
        avgpool_embeddings,
        axis=1,
    ) == 0
)

print("Tous les contrôles des embeddings sont validés.")

## 11. Sauvegarde des features

Les embeddings sont sauvegardés dans `data/processed`.

Les fichiers NumPy sont utilisés pour les calculs rapides, tandis que
les fichiers Parquet offrent un tableau combinant les métadonnées et les
features.

Ces artefacts sont exclus de Git car ils sont directement dérivés des
images médicales.